# 7장 실습 — 환승, 요금, 그리고 지표

6장의 RAPTOR 는 "언제 도착하는가"만 답합니다.
여기에 세 가지를 붙입니다. 환승을 몇 번까지 허용할 것인가, 도보 환승을 몇 미터까지 볼 것인가,
그리고 그 통행이 얼마인가입니다. 교재 7장에 대응합니다.

이 장부터는 교재의 정돈본(`smartmob.teaching.raptor`)을 씁니다.
6장에서 여러분이 짠 것과 같은 알고리즘입니다.

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect, todo
from smartmob.viz import use_korean_font

use_korean_font()

INF = float("inf")

In [ ]:
from smartmob.data import load_gtfs
from smartmob.teaching.raptor import TransitData, journey, raptor, summarize

feed = load_gtfs("hanam")
data = TransitData.from_gtfs(feed)
HANAM_CITY_HALL = (37.5393, 127.2148)
origins = data.access_stops(*HANAM_CITY_HALL)

print(f"정류장 {data.n_stops:,}개, 패턴 {len(data.patterns):,}개, 출발 후보 {len(origins)}곳")

## 1. 요금 규칙 (교재 7.3)

수도권 통합요금은 규칙 세 개로 요약됩니다.

1. 기본요금은 탄 수단 중 가장 비싼 것을 따릅니다
2. 10km 까지는 기본요금만 냅니다
3. 넘으면 5km 마다 100원이 붙습니다 (GTX 를 탔으면 250원)

환승은 요금을 새로 내는 것이 아니라 거리를 이어 붙이는 것입니다.

In [ ]:
from smartmob.teaching.fare import calc_fare, count_transfers, fare_detail

cases = [
    ("버스 한 번 3km", [{"mode": "BUS", "km": 3.0}]),
    ("버스 한 번 15km", [{"mode": "BUS", "km": 15.0}]),
    ("버스 + 지하철 15km", [{"mode": "BUS", "km": 7.0}, {"mode": "SUBWAY", "km": 8.0}]),
    ("걷기만 2km", [{"mode": "WALK", "km": 2.0}]),
]

banner("요금 계산")
for label, legs in cases:
    print(f"{label:22s} {calc_fare(legs):>6,}원  (환승 {count_transfers(legs)}회)")

In [ ]:
fare_detail([{"mode": "BUS", "km": 7.0}, {"mode": "SUBWAY", "km": 8.0}])

버스 따로 지하철 따로 내면 3,050원인데, 통합요금으로는 1,650원입니다.
이 차이가 환승 할인입니다.

## 2. 한 통행을 끝까지 보기

In [ ]:
def hhmm(seconds):
    return "못 감" if seconds == INF else f"{int(seconds) // 3600:02d}:{int(seconds) % 3600 // 60:02d}"


DEPART = 8 * 3600
result = raptor(data, origins, DEPART)

target = data.access_stops(37.5606, 127.1930)[0][0]      # 미사역 근처
legs = journey(data, result, target)

print(f"하남시청 08:00 출발 → {data.stop_names[target]}")
for leg in legs:
    if leg["kind"] == "transit":
        print(f"  {leg['mode']:7s} {leg['route']:10s} "
              f"{hhmm(leg['board_time'])} → {hhmm(leg['alight_time'])}  {leg['km']:.1f}km")
    else:
        print(f"  WALK    {'':10s} {leg['seconds'] / 60:4.1f}분  {leg['km']:.1f}km")

print()
print(summarize(data, legs, DEPART))
print("요금", f"{calc_fare(legs):,}원")

## 3. 지표 여섯 개의 분포 (교재 7.4)

하남시청에서 갈 수 있는 정류장 300곳을 무작위로 골라 지표를 모읍니다.
통행 하나가 아니라 분포를 봐야 이 도시의 대중교통을 판단할 수 있습니다.

In [ ]:
import random

import pandas as pd

rng = random.Random(42)
reachable = [i for i, t in enumerate(result.best) if t < INF]
sample = rng.sample(reachable, min(300, len(reachable)))

rows = []
for stop in sample:
    legs = journey(data, result, stop)
    s = summarize(data, legs, DEPART)
    if not s.get("reachable"):
        continue
    rows.append({
        "총_분": s["total_min"],
        "차내_분": s["in_vehicle_min"],
        "도보_분": s["walk_min"],
        "대기_분": s["wait_min"],
        "환승": s["transfers"],
        "요금": calc_fare(legs),
    })

trips = pd.DataFrame(rows)
trips.describe().round(1)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, col in zip(axes.ravel(), trips.columns):
    ax.hist(trips[col], bins=20, color="#4C6EF5", edgecolor="white")
    ax.set_title(col)
plt.tight_layout();

## 4. 무엇이 시간을 잡아먹는가 (교재 7.5)

총 통행시간을 차내·도보·대기로 나눠 봅니다.
버스에 앉아 있는 시간보다 걷고 기다리는 시간이 길면, 개선의 여지는 노선이 아니라 배차 간격에 있습니다.

In [ ]:
share = trips[["차내_분", "도보_분", "대기_분"]].sum()
share = (share / share.sum() * 100).round(1)

banner("총 통행시간의 구성 (%)")
for name, value in share.items():
    print(f"{name:8s} {value:5.1f}%")

## 5. 환승을 몇 번까지 허용할 것인가 (교재 7.1)

RAPTOR 의 라운드 수가 곧 환승 허용 횟수입니다.
라운드를 늘리면 갈 수 있는 곳이 늘지만, 어느 지점부터는 거의 늘지 않습니다.

In [ ]:
rows = []
for rounds in [1, 2, 3, 4, 5, 6]:
    r = raptor(data, origins, DEPART, max_rounds=rounds)
    reached = sum(1 for t in r.best if t < INF)
    rows.append({
        "라운드": rounds,
        "환승_허용": max(rounds - 1, 0),
        "도달_정류장": reached,
        "도달률": round(reached / data.n_stops, 3),
    })

pd.DataFrame(rows)

## 6. 빈칸

### 6.1 환승을 늘려도 늘지 않는 지점

위 표에서 도달 정류장이 거의 늘지 않기 시작하는 라운드를 찾습니다.
실제 경로 안내 서비스가 환승 횟수에 상한을 두는 이유를 두 줄로 적습니다.

In [ ]:
saturation_round = None     # 도달 정류장이 사실상 늘지 않기 시작하는 라운드

banner("빈칸 6.1")
todo("포화 라운드", saturation_round)

### 6.2 요금이 가장 비싼 통행

`trips` 에서 요금이 가장 비싼 통행의 요금과 그때의 총 통행시간을 찾습니다.
요금과 시간이 비례하는지 산점도로 확인합니다.

In [ ]:
max_fare = None         # 가장 비싼 요금 (원)
fare_time_corr = None   # 요금과 총 통행시간의 상관계수

banner("빈칸 6.2")
todo("가장 비싼 요금", max_fare)
todo("요금과 시간의 상관", fare_time_corr, fmt=lambda v: f"{v:.2f}")

### 6.3 출발 시각을 바꾸면

08:00 대신 22:00 에 출발하면 도달 정류장이 얼마나 줄어드는지 구합니다.
심야에 갈 수 없는 곳이 어디인지가 이 도시 대중교통의 약점입니다.

In [ ]:
night_reached = None    # 22시 출발 시 도달 정류장 수

banner("빈칸 6.3")
todo("22시 도달 정류장", night_reached)

## 정리

- 라운드 수가 환승 허용 횟수입니다. 늘려도 어느 지점부터는 도달 정류장이 늘지 않습니다
- 통합요금은 거리를 이어 붙여 한 번만 냅니다. 환승 할인이 여기서 나옵니다
- 총 통행시간을 차내·도보·대기로 쪼개 보면 무엇을 고쳐야 할지가 보입니다
- 8장 실습에서는 택시 쪽으로 돌아가 수요를 직접 만듭니다